<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Sebastian Raschka《从零构建推理模型》</a>一书的配套代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 6 章：使用强化学习训练推理模型

本 Notebook 使用的程序包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.16
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 LLM 强化学习简介

- 推理时扩展通过为每个生成的答案使用更多计算量来提升推理能力
- 训练时扩展通过在训练期间使用额外计算量来提升推理能力，本章关注的就是这种方法

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- 推理时扩展和训练时扩展也可以（也应该）结合起来，例如在基于 RL 的推理训练之后应用推理时技术
- 实际上，LLM 的 RL 通常作为预训练模型之上的后训练阶段，或作为指令微调之后的阶段来应用

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- 预训练通过预测下一个 token 构建通用知识，而 RL 通过优化序列级目标（例如答案正确性或偏好）来改进模型行为
- LLM 的 RL 包括推理训练和偏好调优，但正如 DeepSeek-R1 所示，面向推理的 RL 也可以直接应用于预训练基础模型
- 直接在基础模型上训练推理能力会得到较弱但仍具备能力的模型（不过这种设置更容易理解推理阶段带来的贡献）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 最初的基于人类反馈的强化学习流程（RLHF）

- RLHF 在 2022 年的 InstructGPT 工作中提出，使用人类偏好标签训练 LLM（这是将 GPT-3 发展为最初 ChatGPT 的关键一步）
- 与优化下一个 token 预测的预训练和监督微调不同，RLHF 根据人类对模型回答的偏好标签来优化模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 从人类反馈到可验证奖励（RLVR）

- RLHF 需要训练一个单独的奖励模型，而它通常是大型且昂贵的 LLM
- RLVR 用自动可验证且确定性的奖励替代学习得到的奖励模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- RLVR 的流行很大程度上源于 DeepSeek-R1 在 2025 年的成功；它展示了无需人类偏好数据或学习得到的奖励模型也能取得强大的推理性能
- DeepSeek-R1 使用自动可验证的奖励训练推理行为，例如检查数学题正确性，以及编程任务的代码编译或执行结果
- 本书聚焦数学验证，但底层思想与代码验证类似：使用二元成功信号自动计算奖励

&nbsp;
## 6.2 使用 GRPO 实现可验证奖励强化学习：逐步讲解

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- 介绍整体思路并了解 RL 如何融入 LLM 开发周期后，我们将实现 RLVR 来训练推理模型（类似 DeepSeek-R1-Zero，但规模小得多；如果按相当规模运行，GPU 成本会达到数十万美元）
- LLM 的 RL 使用一种称为策略梯度的算法更新待训练的 LLM（在 RL 语境中称为“策略”）
- RLHF 中一种流行的策略梯度算法是近端策略优化（PPO）；我们也可以在 RLVR 中使用同一算法
- 不过，DeepSeek 团队训练 DeepSeek-R1 推理模型时使用了更简单的算法，即组相对策略优化（GRPO，最早用于 DeepSeekMath）
- GRPO 更节省资源，因为 PPO 需要另一个 LLM 计算价值函数；GRPO 不需要价值模型，而是从一组采样回答的相对比较中得到学习信号
- 对 PPO 与 GRPO 的详细对比，感兴趣的读者可以参阅我的文章 [The State of Reinforcement Learning for LLM Reasoning](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training)
- 本章使用 GRPO 实现 RLVR
- 此外，下一章将介绍 GRPO 的额外改进，以提升训练稳定性和最终模型性能

### 6.2.1 通过厨师类比建立 GRPO 的直观理解

- GRPO 初看可能比较复杂，因此我想先用“厨师与烹饪”的类比，从整体角度介绍术语并建立一些直觉

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- 在大多数 LLM 强化学习语境中，rollout 和 completion 这两个术语可以互换使用

### 6.2.2 GRPO 的高层流程

- 以下各节实现 GRPO 的技术路线：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 加载预训练模型

- 本章代码与前面章节中的代码相同

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600>

In [2]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{58}

&nbsp;
## 6.4 加载 MATH 训练子集

- 我们使用从原始 MATH 数据集派生的、与评估数据不重叠的训练子集，并明确排除了前几章用于模型评估的 MATH-500 样本（数据集准备方式请参阅 https://github.com/rasbt/math_full_minus_math500）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- 下面的 `load_math_train` 函数与第 3 章的 [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) 函数类似，只是指定了不同的文件路径

In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [5]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

- 注意，我们只需要 `"answer"` 和 `"problem"` 字段
- 理论上可能会想使用 `"solution"`，但这里希望模型自由探索解法（而不是学习某一种特定解答和风格）

&nbsp;
## 6.5 采样 rollouts

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- rollout 是 RL 中对生成回答的术语

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- 我们需要 `@torch.no_grad`，因为不希望为此构建计算图并执行反向传播；但 `@torch.inference_mode` 不可用（它做得太多），会导致

> RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [7]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for step in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- 这里没有新内容
- 上面的代码只是我们之前逐步构建的代码的精简版本：它将第 2 章的 [generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57) 函数与第 4 章的温度和 top-p 采样直接结合起来
- 我们现在不再逐个 yield token，而是将 token 收集到一个张量中，因为不需要实时打印生成的 token

In [8]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 \boxed{58}<|endoftext|>


In [9]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 Let's solve the problem step by step.

**Given:**
\[
\text{Half the value of } 3x - 9 \text{ is } x + 37.
\]

**Step 1: Translate the statement into an equation.**
\[
\frac{1}{2} (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**
\[
3x - 9 = 2(x + 37)
\]

**Step 3: Distribute the 2 on the right side.**
\[
3x - 9 = 2x + 74
\]

**Step 4: Subtract \(2x\) from both sides to get the \(x\)-terms on one side.**
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

**Step 5: Add 9 to both sides to solve for \(x\).**
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}
\]<|endoftext|>


- 实际应用中，我们会多次调用 `sample_response` 来生成 rollouts
- 为了让 GRPO 讲解更简单并与图 6.13 保持一致，这里假设模型生成了下面四个回答：

In [10]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 计算奖励

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- 奖励就是正确性奖励，与第 3 章类似
- 不过还存在隐含的格式奖励：只有当最终答案采用 `\boxed{}` 格式时（通过 `fallback=None`），才会给出 1.0 的奖励

In [11]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [12]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



- 注意：DeepSeek-R1 团队曾尝试使用过程奖励模型，在训练模型时为中间解题步骤打分
- 但这些尝试没有成功，研究人员得出结论：最好只使用最终答案的正确性奖励进行训练，不使用中间奖励

&nbsp;
## 6.7 通过优势值从 rollouts 准备学习信号

- GRPO 中的“GR”（group relative，组相对）指的是：GRPO 针对每个提示词生成多个回答（rollouts），并在它们之间进行相对比较来构造学习信号

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- 公式很简单：

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

- 这里 $r_i$ 表示第 $i$ 个 rollout 的奖励，$\mu_r$ 是该 rollout 组的平均奖励，$\sigma_r$ 是相应的标准差，$\epsilon$ 是为保证数值稳定、避免除零错误而加入的小常数

In [13]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.])


In [14]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


- 注意，如果一组中的所有奖励都相同，例如全为 0 或全为 1，那么对于所有 $i$，都有 $r_i - \mu_r = 0$
- 这意味着当所有答案都正确或全部错误时，模型不会更新

&nbsp;
## 6.8 使用序列对数概率评估 rollouts

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400>

- 上一章实现了 `avg_logprob_answer` 函数，用于计算答案 token 的逐 token 对数概率
- 这些平均对数概率也常称为 token 级对数概率，通常用于给 LLM 回答评分
- 评分时倾向于取平均，因为这样可以进行长度归一化
- 数学上可表示为 $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$
- 这里 $y_1, ..., y_T$ 表示长度为 $T$ 的生成回答中的 token，$y_{<t}$ 表示此前生成的所有 token，$x$ 是输入提示词，$W$ 是模型权重参数
- 这个表达式在数学上与上一章使用的表达式相同；这里只是将 $x$ 换成 $y$，以清楚区分生成输出 token 与输入提示词
- 作为参考，下面复制该函数

In [15]:
## Chapter 5

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

-0.061279296875


- 不过，GRPO 使用的是序列级对数概率，而不是上面这种经过长度归一化的 token 级平均值
- token 级平均值适合评分，因为它能比较不同长度的输出
- 在 GRPO 中，每个 rollout 获得一个针对整个序列的奖励和优势值；为了正确缩放梯度，对数概率必须反映完整序列的似然，而这需要将 token 级对数概率相加
- 否则，对数概率取平均会隐式地按序列长度重新缩放学习信号，扭曲策略更新，尤其会影响较长的 rollout

- 去掉平均操作，并将 `torch.mean(next_token_logps)` 换成 `torch.sum(next_token_logps)`，就可以把它转换为序列级对数概率
- 事后看，也可以将平均结果乘以答案 token 数，得到未平均的值

In [17]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-16.239013671875


- 这些序列级 logprob 随序列长度 $T$ 线性变化
- 这意味着答案越长，其 logprob 总是越负
- 因此对于两个同样好的答案，模型会倾向于更短的答案（因为代价更低）
- 对 logprob 求和会鼓励模型更早停止

- 因此，如上所述，可以在上面的函数中用 `torch.sum` 替换 `torch.mean`
- 不过，由于上一章使用 `@torch.inference_mode()` 装饰器以推理模式运行该函数，而现在希望 PyTorch 跟踪并计算梯度，所以无论如何都必须重新定义它
- 另外，由于第 6.5 节的 `sample_response` 已经返回 `token_ids` 和 `prompt_len`，可以省略 `avg_logprob_answer` 中的编码和 `full_ids` 计算，从而简化函数

In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


- 注意，在 `torch.sum(next_token_logps)` 中我们没有使用 `.item()`，这样 PyTorch 会返回张量（而不是 Python 浮点数），这对梯度计算很重要
- 可以看到，结果值（-16.2998）与之前将 `avg_logprob_val` 乘以答案 token 数得到的值（-16.2390）几乎相同；细微差异可归因于浮点数舍入行为

- 下面将使用 `torch.gather` 重写该函数，这在 PyTorch 中更符合惯用写法，并且在 GPU 上优化得稍好
- 不过，两个函数在数学上是等价的

In [19]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -7.9243

Answer:  The correct answer is \boxed{83}
Logprob: -20.1546

Answer:  The final answer is 83
Logprob: -16.6130

Answer:  We get \boxed{38}
Logprob: -23.3677



- 这里的趋势是：更短、更简洁的回答会获得更高（负值更小）的序列级对数概率
- 唯一包含错误数值的回答（38 而不是 83）获得最低分
- 总体而言，对 logprob 求和会偏好简洁且正确的输出

&nbsp;
## 6.9 从优势值到策略更新：GRPO 损失

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [21]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)


In [22]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-2.5764, grad_fn=<NegBackward0>)


- 需要 `.detach()`，因为我们希望将 `advantages` 当作固定的学习信号；这样可以确保只对 logprob 进行反向传播
- 需要负号，因为 PyTorch 优化器默认执行最小化，而这里希望最大化加权优势的 logprob

- 用数学符号表示，策略梯度损失如下：

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$ 表示批次中的 rollout 数量
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$ 是第 $i$ 个生成回答（长度为 $T_i$）中的 token
- $y_{<t}^{(i)}$ 表示该回答中此前生成的所有 token
- $x^{(i)}$ 是第 $i$ 个 rollout 对应的输入提示词
- $p_W$ 表示模型的策略，即由权重 $W$ 参数化的下一个 token 概率分布
- $A_i$ 是分配给完整第 $i$ 个 rollout 的优势值
- 内层求和计算一个 rollout 的序列级对数概率
- 外层平均计算所有 rollout 上按优势加权的对数概率

&nbsp;
## 6.10 在一个 GRPO 步骤中整合全部内容

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [23]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- 代码注释中的各阶段与 GRPO 图中的阶段对应
- 注意，阶段 1 之后，代码注释中是阶段 2 和 4（而不是 3 和 4），因为这样实现更简单（无需实现多个 for 循环）

In [24]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.

&nbsp;
## 6.11 实现 GRPO 训练循环

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- 由于资源需求已经很高，这里跳过批处理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [25]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- 除了阶段 4（GRPO 损失计算）外，其余部分都是训练深度神经网络（包括 LLM）时的标准训练循环
- `append_csv_metrics` 将结果记录到 CSV 文件中，便于留档（并在第 7 章可视化结果）
- 关于使用 PyTorch 训练神经网络的一般介绍，请参阅我的文章 [PyTorch 一小时入门：从张量到在多 GPU 上训练神经网络](https://sebastianraschka.com/teaching/pytorch-1h/) 的第 3–8 节

- 请注意，如果使用 PyTorch 2.13 而不是 2.7 运行代码，从第 5 步开始会出现细微的数值差异

In [26]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=88.0
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=7.5
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.5
[Step 4/50] loss=0.0909 reward_avg=0.250 avg_resp_len=6.5
[Step 5/50] loss=1.1001 reward_avg=0.500 avg_resp_len=300.5
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth

KeyboardInterrupt. Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00006-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

- 如果运行上面的代码遇到内存问题，可以降低 rollout 数量（例如 `num_rollouts=2`）和每个 rollout 的 token 数（例如 `max_new_tokens=128`）
- 但要获得相对较好的模型，至少需要 `num_rollouts=8` 和 `max_new_tokens=512`
- 如果现有硬件无法运行，也不用担心，下一节会展示如何下载预训练检查点

- 无论如何，这段代码很可能运行得非常慢，因为 GRPO 是资源密集型过程
- 可以随时中断运行，最新模型检查点会保存到 `checkpoints` 文件夹
- 如果想使用云 GPU，请参阅 [GPU 云资源](../../ch02/02_setup-tips/gpu-instructions.md) 文档中的建议

- 注意，这段代码不支持批量训练
- 这是有意的选择，目的是让代码更简单、更易读，同时采样多个（可能很长的）rollout 本身就可能非常耗费资源
- 不过，如果你能使用多块 GPU，可以采用补充材料 [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro) 中提供的、支持批处理和多 GPU 的可选版本，它能更快地训练模型

&nbsp;
## 6.12 加载并评估已保存的模型检查点

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- 可以使用第 2 章介绍的 `model.load_state_dict(torch.load(model_path))` 加载保存的检查点，其中 `model_path` 指向检查点的 “.pth” 文件
- 这些检查点文件也与第 3 章的模型评估工具兼容
- 为方便起见，可以使用第 3 章补充材料中的评估脚本：

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```
    

- 如果不想在自己的电脑上运行 GRPO 训练（因为耗时太久），也可以下载我上传到 [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl) 的检查点（点击要下载的检查点文件，再点击 [下载](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true) 按钮）
- 也可以使用 Python 直接下载检查点

In [27]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

✓ qwen3-0.6B-rlvr-grpo-step00050.pth already up-to-date


|      | 方法                                 | 步数 | 最大 token 数 | Rollout 数 | MATH-500 准确率 | 平均 token 数 |
| ---- | -------------------------------------- | ---- | ---------- | ------------ | ------------ | ------------ |
| 1    | 基础模型（第 3 章）                       | -    |            |              | 15.2%        | 78.85           |
| 2    | 推理模型（第 3 章）                  | -    |            |              | 48.2%        | 1369.79         |
| 3    | GRPO 原始版本但不含 KL（本章） | 50   | 512        | 8            | 47.4%        | 586.11          |

- 从上表可以看到，仅训练 50 步后，以基础模型（第 1 行）初始化的训练模型（第 3 行）已经几乎与原始推理模型（第 2 行）一样好
- 注意，训练更久不一定能提升模型，甚至可能使其变差，因为 GRPO 可能不太稳定；下一章会介绍改进 GRPO 算法的额外技巧

&nbsp;
## 6.13 总结

- 本节没有代码